In [ ]:
%load_ext autoreload
%autoreload 2
# %matplotlib widget
%pdb off

from matplotlib import pyplot as plt
from matplotlib.lines import Line2D
import numpy as np
import pandas as pd
from IPython.display import display, HTML
import pyafn
from pyafn import rho, Cd
import random
from scipy.stats import lognorm
from scipy.stats import norm
from scipy.optimize import curve_fit
from emulationHelpers import readEmulationMI, plot_ventilation_model_fit, plot_empirical_model_error_distribution
import seaborn as sns
from flowEmulationUtils import aggregate_window_series_to_room

#close all figures
plt.close('all')
plt.rcParams['figure.dpi'] = 140
im_scaling = .75
plt.rcParams['figure.figsize'] = [6.4 * im_scaling, 4.8 * im_scaling]
plt.rcParams['font.family'] = 'Times New Roman'

home_dir = "./"
display(home_dir)


In [ ]:
flowStatsMI, roomVentilationMI = readEmulationMI(home_dir=home_dir)

y_var = "flux-Norm"
x_var = "p-noInt_optp0-q_model-Norm"
room_type_order = ["cross", "corner", "dual", "single"]
sigma_q_label = "$\\sigma_{q_n}$ Bulk Fit PN"
sigma_p_label = "$\\sigma_{\\Delta C_p}$ Bulk Fit PN"
model_order = [sigma_q_label, sigma_p_label]
model_tick_labels = {
    sigma_q_label: "$\\sigma_{q_n}$",
    sigma_p_label: "$\\sigma_{\\Delta C_p}$",
}


In [ ]:
# Row 1: baseline q model (categorical hue)
fig, ax, xAdjusted, fittedParams = plot_ventilation_model_fit(
    data=flowStatsMI,
    y_var=y_var,
    x_var=x_var,
    x_var2=None,
    hue="roomType",
    style="slAll",
    model_func=pyafn.ventilationReDecomp_q,
    model_name="$\sigma_{q_n}$ Bulk Fit PN",
    p0=[1.0, 0.1],
    bounds=([0.1, 0], [np.inf, np.inf]),
    adjustData=True,
    show_assymptotes=True,
    add_numeric_colorbar=False,
    return_data=True,
    return_params=True
)

flowStatsMI["q_model-Norm-Adjusted"] = xAdjusted

## ASHRAE Ventilation Rates

In [ ]:
windowASHRAE = pd.read_csv(f"{home_dir}/windowASHRAE.csv")
data=flowStatsMI

ashrae_lookup = windowASHRAE.set_index(["windowType", "roomA", "C"])["ventilationRate"]
data["ASHRAE"] = data.apply(
    lambda row: ashrae_lookup.get((row["windowType"], row["roomA"], row["C"]), np.nan) if not row["slAll"] else np.nan, axis=1
)

plot_df = data[data["slAll"] == False].copy()

fig, axs = plt.subplots(1, 3, figsize=(12, 4.8), dpi=160, sharey=True, layout="constrained")

scatter_specs = [
    ("ASHRAE", "ASHRAE Pressures",  "Modeled Flux Velocity"),
    (x_var, "LES Pressures", "Modeled Flux Velocity"),
    (xAdjusted, "LES Pressures", "Fluctuation-Adjusted Flux Velocity"),
]

for ax, (xcol, title, xlabel) in zip(axs, scatter_specs):
    sns.scatterplot(
        data=plot_df,
        x=xcol,
        y=y_var,
        hue="roomType",
        # style="roomType",
        palette="colorblind",
        hue_order=room_type_order,
        alpha=0.65,
        s=20,
        ax=ax,
        legend=ax is axs[0],
    )

    lim_min = np.nanmin([plot_df[x_var].min(), plot_df[y_var].min()])
    lim_max = np.nanmax([plot_df[x_var].max(), plot_df[y_var].max()])
    ax.plot(
        [lim_min, lim_max],
        [lim_min, lim_max],
        "r--",
        linewidth=1.2,
        alpha=0.7,
        label="1:1" if ax is axs[0] else None,
    )

    ax.set_title(title, fontsize=18)
    ax.set_xlabel(xlabel, fontsize=16)
    ax.set_ylabel("LES Flux Velocity", fontsize=16)
    ax.grid(True, alpha=0.3)
    ax.tick_params(labelsize=14)
    # ax.set_xlim(-0.6, 0.6)
    # ax.set_ylim(-0.6, 0.6)


handles, labels = axs[0].get_legend_handles_labels()
for ax in axs:
    if ax.get_legend() is not None:
        ax.get_legend().remove()

# fig.legend(
#     handles,
#     labels,
#     loc="center left",
#     bbox_to_anchor=(0.90, 0.5),
#     fontsize=12,
#     # title="Room A / Window Type",
#     title_fontsize=13,
#     frameon=False,
# )

fig.suptitle("Window-Level Ventilation Comparison", fontsize=20)
fig.subplots_adjust(left=0.08, right=0.86, top=0.83, bottom=0.18, wspace=0.30)

In [ ]:
data_WA_mean = data[data["slAll"] == False].groupby(["windowType", "roomA"])[y_var].mean().reset_index()

windowTypeOrder = data_WA_mean["windowType"].dropna().unique()
plt.figure()
sns.lineplot(data=data_WA_mean, x="roomA", y=y_var, hue="windowType", palette="tab10", hue_order=windowTypeOrder)
sns.lineplot(data=windowASHRAE, x="roomA", y="ventilationRate", hue="windowType", palette="tab10", linestyle='--', hue_order=windowTypeOrder)

In [ ]:
roomVentilationMI_adj = aggregate_window_series_to_room(
    roomVentilationMI, xAdjusted, out_col="xAdjusted_room", mode="abs_half_sum"
)

In [ ]:
roomASHRAE = pd.read_csv(f"{home_dir}/roomASHRAE.csv")

ashrae_lookup = roomASHRAE.set_index(["roomType", "AofA", "C"])["ventilationRate"]
roomVentilationMI_adj["ASHRAE"] = roomVentilationMI_adj.apply(
    lambda row: ashrae_lookup.get((row["roomType"], row["AofA"], row["C"]), np.nan) if not row["slAll"] else np.nan, axis=1
)

plt.figure()
sns.scatterplot(data=roomVentilationMI_adj[roomVentilationMI_adj["slAll"] == False], x="ASHRAE", y=y_var, hue="AofA", palette="viridis")

## New Plots

In [ ]:
fig, ax, sigma_q_adjusted, sigma_q_fitted_params = plot_ventilation_model_fit(
    data=flowStatsMI,
    y_var=y_var,
    x_var=x_var,
    x_var2=None,
    hue="roomType",
    style="slAll",
    model_func=pyafn.ventilationReDecomp_q,
    model_name=sigma_q_label,
    p0=[1.0, 0.1],
    bounds=([0.1, 0], [np.inf, np.inf]),
    adjustData=True,
    show_assymptotes=True,
    add_numeric_colorbar=False,
    return_data=True,
    return_params=True
)
plt.close(fig)

fig, ax, sigma_p_adjusted, sigma_p_fitted_params = plot_ventilation_model_fit(
    data=flowStatsMI,
    y_var=y_var,
    x_var=x_var,
    x_var2=None,
    hue="roomType",
    style="slAll",
    model_func=pyafn.ventilationReDecomp_p,
    model_name=sigma_p_label,
    p0=[1.0, 0.01],
    bounds=([0.1, 0], [np.inf, np.inf]),
    adjustData=True,
    show_assymptotes=True,
    add_numeric_colorbar=False,
    return_data=True,
    return_params=True
)
plt.close(fig)

flowStatsMI["sigma_q_bulk_fit_pn"] = sigma_q_adjusted
flowStatsMI["sigma_p_bulk_fit_pn"] = sigma_p_adjusted


## ASHRAE PN Bulk-Fit Comparison

In [ ]:
windowASHRAE = pd.read_csv(f"{home_dir}/windowASHRAE.csv")
roomASHRAE = pd.read_csv(f"{home_dir}/roomASHRAE.csv")

def attach_window_ashrae(frame, ashrae_frame):
    ashrae_lookup = ashrae_frame.set_index(["windowType", "roomA", "C"])["ventilationRate"]
    out = frame.copy()
    out["ASHRAE"] = out.apply(
        lambda row: ashrae_lookup.get((row["windowType"], row["roomA"], row["C"]), np.nan)
        if not row["slAll"] else np.nan,
        axis=1,
    )
    return out

def attach_room_ashrae(frame, ashrae_frame):
    ashrae_lookup = ashrae_frame.set_index(["roomType", "AofA", "C"])["ventilationRate"]
    out = frame.copy()
    out["ASHRAE"] = out.apply(
        lambda row: ashrae_lookup.get((row["roomType"], row["AofA"], row["C"]), np.nan)
        if not row["slAll"] else np.nan,
        axis=1,
    )
    return out

def compute_error_metrics(frame, pred_col, group_col, model_name):
    rows = []
    group_specs = [("total", "All", pd.Series(True, index=frame.index))]
    for group in frame[group_col].dropna().unique():
        group_specs.append(("group", group, frame[group_col] == group))

    for scope, group, mask in group_specs:
        subset = frame.loc[mask, ["ASHRAE", pred_col]].dropna()
        if subset.empty:
            continue

        error = subset[pred_col] - subset["ASHRAE"]
        rmse = np.sqrt(np.mean(error ** 2))
        ashrae_std = np.std(subset["ASHRAE"])
        nrmse = rmse / ashrae_std if ashrae_std > 0 else np.nan
        rows.append(
            {
                "model_name": model_name,
                "scope": scope,
                "group": group,
                "rmse": rmse,
                "bias": error.mean(),
                "nrmse": nrmse,
                "n": len(subset),
            }
        )

    return pd.DataFrame(rows)

def _plot_point(ax, y_pos, value, color):
    if pd.isna(value):
        return
    ax.scatter(
        value,
        y_pos,
        color=color,
        edgecolor="black",
        linewidth=0.6,
        s=42,
        zorder=3,
    )

def _plot_connector(ax, y_pos, values):
    clean_values = [value for value in values if pd.notna(value)]
    if len(clean_values) < 2:
        return
    ax.plot(
        clean_values,
        [y_pos] * len(clean_values),
        color="0.35",
        linewidth=1.2,
        alpha=0.25,
        zorder=2,
    )

def plot_error_summary(metrics_df, title, group_palette, model_order, model_tick_labels):
    metric_rows = [("rmse", "RMSE"), ("bias", "Bias")]
    fig, axs = plt.subplots(2, 1, figsize=(8.2, 7.8), dpi=160, sharey=True)
    group_data = metrics_df[metrics_df["scope"] == "group"].copy()
    group_order = [group for group in group_palette if group in group_data["group"].unique()]

    for row_idx, (metric_col, metric_label) in enumerate(metric_rows):
        ax = axs[row_idx]
        for model_idx, model_name in enumerate(model_order):
            point_values = []
            for group in group_order:
                row = group_data[
                    (group_data["model_name"] == model_name)
                    & (group_data["group"] == group)
                ]
                if row.empty:
                    continue
                value = row.iloc[0][metric_col]
                point_values.append(value)
                _plot_point(ax, model_idx, value, group_palette[group])
            _plot_connector(ax, model_idx, point_values)

        ax.axvline(0, color="k", linewidth=0.8, alpha=0.7)
        ax.grid(axis="x", alpha=0.25)
        ax.grid(axis="y", alpha=0.35, linestyle="--")
        ax.set_yticks(range(len(model_order)))
        ax.set_yticklabels([model_tick_labels[name] for name in model_order], fontsize=16)
        ax.tick_params(axis="x", labelsize=14)
        ax.tick_params(axis="y", labelsize=16)
        ax.set_ylim(-0.5, len(model_order) - 0.5)
        ax.set_xlabel(metric_label, fontsize=16)
        if row_idx == 0:
            ax.set_title(title, fontsize=18)

    handles = [
        Line2D([0], [0], color=color, marker="o", linestyle="None", markersize=9, label=group)
        for group, color in group_palette.items()
        if group in group_order
    ]
    fig.legend(
        handles=handles,
        loc="lower center",
        bbox_to_anchor=(0.5, 0.01),
        ncol=max(1, len(handles)),
        fontsize=12,
        frameon=False,
    )
    fig.tight_layout(rect=[0, 0.07, 1, 1])
    return fig, axs

window_plot_df = attach_window_ashrae(flowStatsMI, windowASHRAE)
window_plot_df = window_plot_df[window_plot_df["slAll"] == False].copy()

room_plot_df = aggregate_window_series_to_room(
    roomVentilationMI,
    flowStatsMI["sigma_q_bulk_fit_pn"],
    out_col="sigma_q_bulk_fit_pn",
    mode="abs_half_sum",
)
room_plot_df = aggregate_window_series_to_room(
    room_plot_df,
    flowStatsMI["sigma_p_bulk_fit_pn"],
    out_col="sigma_p_bulk_fit_pn",
    mode="abs_half_sum",
)
room_plot_df = attach_room_ashrae(room_plot_df, roomASHRAE)
room_plot_df = room_plot_df[room_plot_df["slAll"] == False].copy()

window_error_metrics_df = pd.concat(
    [
        compute_error_metrics(window_plot_df, "sigma_q_bulk_fit_pn", "windowType", sigma_q_label),
        compute_error_metrics(window_plot_df, "sigma_p_bulk_fit_pn", "windowType", sigma_p_label),
    ],
    ignore_index=True,
)

room_error_metrics_df = pd.concat(
    [
        compute_error_metrics(room_plot_df, "sigma_q_bulk_fit_pn", "roomType", sigma_q_label),
        compute_error_metrics(room_plot_df, "sigma_p_bulk_fit_pn", "roomType", sigma_p_label),
    ],
    ignore_index=True,
)

window_match_summary = pd.DataFrame(
    {
        "dataset": ["windows"],
        "rows": [len(window_plot_df)],
        "ashrae_matches": [window_plot_df["ASHRAE"].notna().sum()],
    }
)
room_match_summary = pd.DataFrame(
    {
        "dataset": ["rooms"],
        "rows": [len(room_plot_df)],
        "ashrae_matches": [room_plot_df["ASHRAE"].notna().sum()],
    }
)


In [ ]:
window_type_order = list(window_plot_df["windowType"].dropna().unique())
window_palette = dict(zip(window_type_order, sns.color_palette("tab10", n_colors=len(window_type_order))))
room_palette = {
    room_type: color
    for room_type, color in zip(room_type_order, sns.color_palette("colorblind", n_colors=len(room_type_order)))
    if room_type in room_plot_df["roomType"].dropna().unique()
}

scatter_specs = [
    (sigma_q_label, "sigma_q_bulk_fit_pn"),
    (sigma_p_label, "sigma_p_bulk_fit_pn"),
]

scatter_limit_values = [
    window_plot_df["ASHRAE"],
    room_plot_df["ASHRAE"],
    window_plot_df["sigma_q_bulk_fit_pn"],
    window_plot_df["sigma_p_bulk_fit_pn"],
    room_plot_df["sigma_q_bulk_fit_pn"],
    room_plot_df["sigma_p_bulk_fit_pn"],
]
scatter_limit_values = np.concatenate([series.dropna().to_numpy() for series in scatter_limit_values if not series.dropna().empty])
scatter_min = np.nanmin(scatter_limit_values)
scatter_max = np.nanmax(scatter_limit_values)

fig_fit, axs_fit = plt.subplots(2, 2, figsize=(12.5, 9.5), dpi=160, sharex=True, sharey=True)

for col_idx, (title, pred_col) in enumerate(scatter_specs):
    sns.scatterplot(
        data=window_plot_df,
        x="ASHRAE",
        y=pred_col,
        hue="windowType",
        palette=window_palette,
        hue_order=window_type_order,
        alpha=0.75,
        s=26,
        ax=axs_fit[0, col_idx],
        legend=col_idx == 0,
    )
    axs_fit[0, col_idx].plot([scatter_min, scatter_max], [scatter_min, scatter_max], "r--", linewidth=1.2, alpha=0.7)
    axs_fit[0, col_idx].set_title(title, fontsize=18)
    axs_fit[0, col_idx].set_ylabel("Window Prediction", fontsize=16)
    axs_fit[0, col_idx].grid(True, alpha=0.3)
    axs_fit[0, col_idx].tick_params(labelsize=13)
    axs_fit[0, col_idx].set_xlim(scatter_min, scatter_max)
    axs_fit[0, col_idx].set_ylim(scatter_min, scatter_max)

    sns.scatterplot(
        data=room_plot_df,
        x="ASHRAE",
        y=pred_col,
        hue="roomType",
        palette=room_palette,
        hue_order=[room for room in room_type_order if room in room_palette],
        alpha=0.75,
        s=26,
        ax=axs_fit[1, col_idx],
        legend=col_idx == 0,
    )
    axs_fit[1, col_idx].plot([scatter_min, scatter_max], [scatter_min, scatter_max], "r--", linewidth=1.2, alpha=0.7)
    axs_fit[1, col_idx].set_xlabel("ASHRAE Ventilation Rate", fontsize=16)
    axs_fit[1, col_idx].set_ylabel("Room Prediction", fontsize=16)
    axs_fit[1, col_idx].grid(True, alpha=0.3)
    axs_fit[1, col_idx].tick_params(labelsize=13)
    axs_fit[1, col_idx].set_xlim(scatter_min, scatter_max)
    axs_fit[1, col_idx].set_ylim(scatter_min, scatter_max)

for ax in [axs_fit[0, 1], axs_fit[1, 1]]:
    if ax.get_legend() is not None:
        ax.get_legend().remove()

axs_fit[0, 0].set_title("$\\sigma_{q_n}$ Bulk Fit PN", fontsize=18)
axs_fit[0, 1].set_title("$\\sigma_{\\Delta C_p}$ Bulk Fit PN", fontsize=18)
axs_fit[0, 0].text(-0.26, 0.5, "Windows", rotation=90, transform=axs_fit[0, 0].transAxes, va="center", ha="center", fontsize=18)
axs_fit[1, 0].text(-0.26, 0.5, "Rooms (Skylights Closed)", rotation=90, transform=axs_fit[1, 0].transAxes, va="center", ha="center", fontsize=18)
fig_fit.suptitle("ASHRAE PN Bulk-Fit Model Comparison", fontsize=20)
fig_fit.tight_layout(rect=[0.04, 0, 1, 0.96])


In [ ]:
display(window_match_summary)
display(room_match_summary)
display(window_error_metrics_df)
display(room_error_metrics_df)


In [ ]:
fig_window_error, axs_window_error = plot_error_summary(
    window_error_metrics_df,
    "Window Error Summary vs ASHRAE",
    window_palette,
    model_order,
    model_tick_labels,
)

fig_room_error, axs_room_error = plot_error_summary(
    room_error_metrics_df,
    "Room Error Summary vs ASHRAE",
    room_palette,
    model_order,
    model_tick_labels,
)
